In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (4).jpg
/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (5).jpg
/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (8).jpg
/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (3).jpg
/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (10).jpg
/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (1).jpg
/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (9).jpg
/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (7).jpg
/kaggle/input/datasets/abdulhas

In [3]:
# Install albumentations if it's not already on the latest version
!pip install -q albumentations

import os
import cv2
import torch
import numpy as np
import pandas as pd
import albumentations as A
import matplotlib.pyplot as plt
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torch.nn as nn
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
import os

# Update these to match the exact input paths in your Kaggle environment
train_dir = '/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train' # or wherever your dataset unzips
test_dir = '/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test'

# 1. Get consistent class names across both train and test sets
# (Ensures class index 0 is always 'Apple Scab Leaf' in both sets)
class_names = sorted(os.listdir(test_dir))
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

def get_image_paths_and_labels(data_dir):
    paths = []
    labels = []
    for class_name in os.listdir(data_dir):
        class_dir = os.path.join(data_dir, class_name)
        if os.path.isdir(class_dir) and class_name in class_to_idx:
            class_idx = class_to_idx[class_name]
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    paths.append(os.path.join(class_dir, img_name))
                    labels.append(class_idx)
    return paths, labels

# Load Train and Test image paths & labels directly
train_paths, train_labels = get_image_paths_and_labels(train_dir)
val_paths, val_labels = get_image_paths_and_labels(test_dir)

print(f"Total Classes: {len(class_names)}")
print(f"Training Images: {len(train_paths)}")
print(f"Validation/Test Images: {len(val_paths)}")

Total Classes: 27
Training Images: 2316
Validation/Test Images: 236


In [5]:
IMAGE_SIZE = 224

# Heavy augmentation for messy field images (Training)
train_transform = A.Compose([
    # Resizing strategy: resize shorter side to 256, then crop to 224x224
    A.SmallestMaxSize(max_size=256),
    A.RandomCrop(height=IMAGE_SIZE, width=IMAGE_SIZE),
    
    # Flips & Rotations
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
    
    # Lighting, Contrast & Color distortions
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    
    # ImageNet Normalization and Tensor Conversion
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Deterministic pipeline for evaluation (Validation/Test)
val_transform = A.Compose([
    A.SmallestMaxSize(max_size=256),
    A.CenterCrop(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [12]:
class PlantDocDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        # Load image via OpenCV
        image = cv2.imread(img_path)
        
        # Handle unreadable/corrupt files safely
        if image is None:
            image = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
        # Apply Albumentations pipeline
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
            
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return image, label

# Instantiate Datasets
train_dataset = PlantDocDataset(train_paths, train_labels, transform=train_transform)
val_dataset = PlantDocDataset(val_paths, val_labels, transform=val_transform)

# Build DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 37 | Val batches: 4


In [17]:
import torchvision.models as models
import torch.nn as nn
import torch

# ==========================================
# 1. MODEL SETUP (Upgraded to ResNet50 + Dropout + Weight Decay)
# ==========================================
# Load pretrained ResNet50
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze lower convolutional layers (Stage 1 Setup)
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier layer with a Sequential block containing Dropout
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5), # Drops 50% of connections to prevent overfitting
    nn.Linear(num_features, len(class_names))
)

# Send model to GPU if available
model = model.to(device)

# Loss function and optimizer for STAGE 1 (updating ONLY fc parameters)
criterion = nn.CrossEntropyLoss()
# ADDED weight_decay here for L2 Regularization
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001, weight_decay=1e-4) 

# Initialize the scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [18]:
# ==========================================
# 2. TWO-STAGE TRAINING LOOP
# ==========================================
TOTAL_EPOCHS = 30
UNFREEZE_EPOCH = 5 # Start fine-tuning at epoch 5

best_val_acc = 0.0
SAVE_PATH = '/kaggle/working/best_plantdoc_model.pth'

for epoch in range(TOTAL_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{TOTAL_EPOCHS} ---")
    
    # --- TWO-STAGE TRIGGER ---
    if epoch == UNFREEZE_EPOCH:
        print("\n[INFO] STAGE 2 INITIATED: Unfreezing base layers for fine-tuning!")
        # Unfreeze all layers
        for param in model.parameters():
            param.requires_grad = True
            
        # Re-initialize optimizer to train ALL parameters with a smaller learning rate
        # ADDED weight_decay here as well
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4) 
        
        # Re-initialize scheduler for the new optimizer
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    
    # ================= TRAINING =================
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in tqdm(train_loader, desc="Training Batch"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (preds == labels).sum().item()
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    epoch_train_acc = 100.0 * correct_train / total_train
    
    # ================= VALIDATION =================
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation Batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (preds == labels).sum().item()
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    epoch_val_acc = 100.0 * correct_val / total_val
    
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.2f}%")
    
    # Step the scheduler based on validation loss
    scheduler.step(epoch_val_loss)
    
    # Track best model checkpoint
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f" Saved new best model checkpoint to {SAVE_PATH}! (Val Acc: {best_val_acc:.2f}%)")


--- Epoch 1/30 ---


Validation Batch: 100%|██████████| 4/4 [00:02<00:00,  1.48it/s]


Train Loss: 2.9535 | Train Acc: 21.85%
Val Loss:   2.7367 | Val Acc:   30.08%
 Saved new best model checkpoint to /kaggle/working/best_plantdoc_model.pth! (Val Acc: 30.08%)

--- Epoch 2/30 ---


Validation Batch: 100%|██████████| 4/4 [00:02<00:00,  1.52it/s]


Train Loss: 2.3827 | Train Acc: 40.24%
Val Loss:   2.3739 | Val Acc:   38.56%
 Saved new best model checkpoint to /kaggle/working/best_plantdoc_model.pth! (Val Acc: 38.56%)

--- Epoch 3/30 ---


...

Train Loss: 0.1693 | Train Acc: 94.86%
Val Loss:   1.0932 | Val Acc:   66.95%

--- Epoch 29/30 ---


Validation Batch: 100%|██████████| 4/4 [00:02<00:00,  1.51it/s]


Train Loss: 0.1708 | Train Acc: 95.16%
Val Loss:   1.0947 | Val Acc:   66.53%

--- Epoch 30/30 ---


Validation Batch: 100%|██████████| 4/4 [00:02<00:00,  1.48it/s]


Train Loss: 0.1615 | Train Acc: 95.25%
Val Loss:   1.1121 | Val Acc:   66.95%


In [19]:
save_path = 'plantdoc_resnet18_final.pth'
torch.save(model.state_dict(), save_path)
print(f"Final model weights saved to {save_path}")

Final model weights saved to plantdoc_resnet18_final.pth


In [25]:
def predict_leaf_disease(image_path, model, class_names, transform):
    model.eval()
    
    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    augmented = transform(image=image)
    image_tensor = augmented['image'].unsqueeze(0).to(device) # Add batch dimension
    
    # Perform prediction
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)
        
    predicted_class = class_names[predicted_idx.item()]
    confidence_score = confidence.item() * 100
    
    return predicted_class, confidence_score

# --- Test Inference on an image from the test set ---
sample_test_img = val_paths[1]
true_label = class_names[val_labels[0]]

pred_class, conf = predict_leaf_disease(sample_test_img, model, class_names, val_transform)

print(f"Sample Image Path: {sample_test_img}")
print(f"True Label:      {true_label}")
print(f"Predicted Class: {pred_class} ({conf:.2f}% confidence)")

Sample Image Path: /kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test/Corn leaf blight/Corn leaf blight (5).jpg
True Label:      Corn leaf blight
Predicted Class: Corn leaf blight (99.77% confidence)
